Data validation and schema enforcement
Goal: Validate schema, enforce data types, and handle missing values

In [1]:
# Import libraries and load data
print("="*70)
print("Step 2: Data validation and schema enforcement")
print("="*70)

import pandas as pd
import numpy as np
from pathlib import Path

# File paths
TRAIN_FILE = Path("eurlex.csv")
TEST_FILE = Path("eurlex_test.csv")

# Load datasets
print("[1/5] Loading datasets...")
df_train = pd.read_csv(TRAIN_FILE)
df_test = pd.read_csv(TEST_FILE)

print(f"[1/5] Train shape: {df_train.shape}")
print(f"[1/5] Test shape: {df_test.shape}")

Step 2: Data validation and schema enforcement
[1/5] Loading datasets...
[1/5] Train shape: (15377, 5002)
[1/5] Test shape: (3971, 5001)


In [2]:
# Schema validation
print("[2/5] Validating schema...")

# Check column names
train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

# Test should have all columns except 'labels'
expected_test_cols = train_cols - {'labels'}

if test_cols == expected_test_cols:
    print("[2/5] Test columns match expected schema")
else:
    print("[2/5] Schema mismatch detected!")
    print(f"  Missing in test: {expected_test_cols - test_cols}")
    print(f"  Extra in test: {test_cols - expected_test_cols}")

# Verify feature columns
feature_cols = [col for col in df_train.columns if col.startswith('f')]
print(f"[2/5] Number of feature columns: {len(feature_cols)}")

[2/5] Validating schema...
[2/5] Test columns match expected schema
[2/5] Number of feature columns: 5000


In [3]:
# Data type validation and enforcement
print("[3/5] Validating and enforcing data types...")

# row_id should be integer
df_train['row_id'] = df_train['row_id'].astype(int)
df_test['row_id'] = df_test['row_id'].astype(int)
print("[3/5] row_id converted to int")

# All feature columns should be numeric (float)
for col in feature_cols:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')

print(f"[3/5] All {len(feature_cols)} feature columns converted to numeric")

# labels column should be string
df_train['labels'] = df_train['labels'].astype(str)
print("[3/5] labels column set as string")

print("[3/5] Data types summary:")
print(f"  row_id: {df_train['row_id'].dtype}")
print(f"  features: {df_train[feature_cols[0]].dtype}")
print(f"  labels: {df_train['labels'].dtype}")

[3/5] Validating and enforcing data types...
[3/5] row_id converted to int
[3/5] All 5000 feature columns converted to numeric
[3/5] labels column set as string
[3/5] Data types summary:
  row_id: int64
  features: float64
  labels: object


In [4]:
# Missing value analysis and handling
print("[4/5] Analyzing missing values...")

# Check for missing values in features
train_missing = df_train[feature_cols].isnull().sum().sum()
test_missing = df_test[feature_cols].isnull().sum().sum()

print(f"[4/5] Train missing values in features: {train_missing}")
print(f"[4/5] Test missing values in features: {test_missing}")

if train_missing > 0 or test_missing > 0:
    print("[4/5] Missing values detected! Handling strategy:")
    print("  -> Filling missing values with 0.0 (appropriate for sparse features)")
    # Fill missing values with 0.0
    df_train[feature_cols] = df_train[feature_cols].fillna(0.0)
    df_test[feature_cols] = df_test[feature_cols].fillna(0.0)
    # Verify
    train_missing_after = df_train[feature_cols].isnull().sum().sum()
    test_missing_after = df_test[feature_cols].isnull().sum().sum()
    print(f"[4/5] After filling - Train missing: {train_missing_after}, Test missing: {test_missing_after}")
else:
    print("[4/5] No missing values detected")

# Check for missing labels in train
train_label_missing = df_train['labels'].isnull().sum()
print(f"[4/5] Missing labels in train: {train_label_missing}")

if train_label_missing > 0:
    print("[4/5] Warning: Some training samples have no labels")
    print("  -> These will be assigned empty label set ''")
    df_train['labels'] = df_train['labels'].fillna('')

[4/5] Analyzing missing values...
[4/5] Train missing values in features: 0
[4/5] Test missing values in features: 0
[4/5] No missing values detected
[4/5] Missing labels in train: 0


In [5]:
# Data quality summary report
print("[5/5] Data quality summary report")
print("="*70)

def get_stats(df, name):
    
    print(f"\n{name} dataset:")
    print(f"  Rows: {len(df):,}")
    
    print(f"  Features: {len(feature_cols)}")
    # Feature statistics
    feature_data = df[feature_cols]
    print(f"  Feature value range: [{feature_data.min().min():.4f}, {feature_data.max().max():.4f}]")
    print(f"  Feature mean: {feature_data.mean().mean():.4f}")
    print(f"  Feature std: {feature_data.std().mean():.4f}")
    # Sparsity
    zero_count = (feature_data == 0).sum().sum()
    total_elements = feature_data.size
    sparsity = (zero_count / total_elements) * 100
    print(f"  Sparsity: {sparsity:.2f}% (zeros)")
    if name == "Train":
        # Label statistics
        label_counts = df['labels'].str.split().str.len()
        print(f"  Labels per sample - Mean: {label_counts.mean():.2f}, Max: {label_counts.max()}")

get_stats(df_train, "Train")
get_stats(df_test, "Test")

print("[5/5] Data validation complete!")

[5/5] Data quality summary report

Train dataset:
  Rows: 15,377
  Features: 5000
  Feature value range: [0.0000, 45781.1000]
  Feature mean: 0.4182
  Feature std: 5.4026
  Sparsity: 95.24% (zeros)
  Labels per sample - Mean: 5.31, Max: 24

Test dataset:
  Rows: 3,971
  Features: 5000
  Feature value range: [0.0000, 40277.3000]
  Feature mean: 0.4129
  Feature std: 5.0570
  Sparsity: 95.34% (zeros)
[5/5] Data validation complete!


Step 3: Preprocessing (normalization)
Goal: Apply min-max normalization (0-1 scaling) to all features

Categorical encoding was not required because all input feature columns were already numeric in the original data and remained numeric after conversion to CSV. Only the labels field is non-numeric, and it is treated as the target, not as an input feature.

In [6]:
# Compute normalization parameters from training data
print("="*70)
print("Step 3: Preprocessing - min-max normalization")
print("="*70)

print("[1/4] Computing normalization parameters from training data...")

# Calculate min and max for each feature column from training data ONLY
feature_min = df_train[feature_cols].min()
feature_max = df_train[feature_cols].max()

# Handle constant features (where min == max)
constant_features = (feature_min == feature_max)
num_constant = constant_features.sum()

if num_constant > 0:
    print(f"[1/4] Found {num_constant} constant features (min = max)")
    print("  -> These will remain unchanged during normalization")

print(f"[1/4] Normalization parameters computed for {len(feature_cols)} features")
print(f"  Sample ranges: {feature_cols[0]}: [{feature_min[0]:.4f}, {feature_max[0]:.4f}]")
print(f"                 {feature_cols[100]}: [{feature_min[100]:.4f}, {feature_max[100]:.4f}]")

Step 3: Preprocessing - min-max normalization
[1/4] Computing normalization parameters from training data...
[1/4] Normalization parameters computed for 5000 features
  Sample ranges: f0: [0.0000, 351.4250]
                 f100: [0.0000, 41.1983]


C:\Users\samee\AppData\Local\Temp\ipykernel_36412\2244397066.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"  Sample ranges: {feature_cols[0]}: [{feature_min[0]:.4f}, {feature_max[0]:.4f}]")
C:\Users\samee\AppData\Local\Temp\ipykernel_36412\2244397066.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"                 {feature_cols[100]}: [{feature_min[100]:.4f}, {feature_max[100]:.4f}]")


In [7]:
# Apply Min-Max scaling (0-1 normalization)
print("[2/4] Applying min-max scaling to [0, 1] range...")

def min_max_normalize(df, feature_cols, feat_min, feat_max):
    """
    Apply Min-Max normalization: X_norm = (X - min) / (max - min)
    For constant features (min == max), keep original value
    """
    df_normalized = df.copy()
    for col in feature_cols:
        min_val = feat_min[col]
        max_val = feat_max[col]
        if min_val == max_val:
            # Constant feature - no normalization needed
            df_normalized[col] = df[col]
        else:
            # Standard Min-Max normalization
            df_normalized[col] = (df[col] - min_val) / (max_val - min_val)
    return df_normalized

# Normalize both train and test using TRAIN statistics
df_train_normalized = min_max_normalize(df_train, feature_cols, feature_min, feature_max)
df_test_normalized = min_max_normalize(df_test, feature_cols, feature_min, feature_max)

print("[2/4] Train data normalized")
print("[2/4] Test data normalized (using train min/max)")

[2/4] Applying min-max scaling to [0, 1] range...
[2/4] Train data normalized
[2/4] Test data normalized (using train min/max)


In [8]:
# Verify normalization
print("[3/4] Verifying normalization...")

# Check normalized ranges
train_norm_min = df_train_normalized[feature_cols].min().min()
train_norm_max = df_train_normalized[feature_cols].max().max()
test_norm_min = df_test_normalized[feature_cols].min().min()
test_norm_max = df_test_normalized[feature_cols].max().max()

print(f"[3/4] Train normalized range: [{train_norm_min:.6f}, {train_norm_max:.6f}]")
print(f"[3/4] Test normalized range: [{test_norm_min:.6f}, {test_norm_max:.6f}]")

# Note: test range might slightly exceed [0,1] if test has values outside train range
if test_norm_min < -0.01 or test_norm_max > 1.01:
    print("[3/4] Test data contains values outside training range")
    print("  This is normal if test distribution differs from train")

# Sample comparison
print("[3/4] Sample comparison (first 3 features, first 2 rows):")
print("Before normalization:")
print(df_train[feature_cols[:3]].head(2))
print("After normalization:")
print(df_train_normalized[feature_cols[:3]].head(2))

[3/4] Verifying normalization...
[3/4] Train normalized range: [0.000000, 1.000000]
[3/4] Test normalized range: [0.000000, 44.281421]
[3/4] Test data contains values outside training range
  This is normal if test distribution differs from train
[3/4] Sample comparison (first 3 features, first 2 rows):
Before normalization:
    f0   f1   f2
0  0.0  0.0  0.0
1  0.0  0.0  0.0
After normalization:
    f0   f1   f2
0  0.0  0.0  0.0
1  0.0  0.0  0.0


In [9]:
# Save normalized datasets
print("[4/4] Saving normalized datasets...")

# Clip test values to [0, 1] to handle out-of-range values
print("[4/4] Clipping test features to [0, 1] range...")
df_test_normalized[feature_cols] = df_test_normalized[feature_cols].clip(0, 1)

# Verify after clipping
test_clipped_max = df_test_normalized[feature_cols].max().max()
test_clipped_min = df_test_normalized[feature_cols].min().min()
print(f"[4/4] Test range after clipping: [{test_clipped_min:.6f}, {test_clipped_max:.6f}]")

OUT_TRAIN_NORMALIZED = Path("eurlex_normalized.csv")
OUT_TEST_NORMALIZED = Path("eurlex_test_normalized.csv")

df_train_normalized.to_csv(OUT_TRAIN_NORMALIZED, index=False)
df_test_normalized.to_csv(OUT_TEST_NORMALIZED, index=False)

[4/4] Saving normalized datasets...
[4/4] Clipping test features to [0, 1] range...
[4/4] Test range after clipping: [0.000000, 1.000000]


Testing preprocessing on sample data

In [10]:
# Test preprocessing on sample data
print("="*70)
print("Testing preprocessing on sample data")
print("="*70)

# Take a small sample for testing
sample_size = 100
df_train_sample = df_train_normalized.head(sample_size).copy()

print(f"[1/3] Created sample with {sample_size} rows")

# Extract features and labels
X_sample = df_train_sample[feature_cols].values
y_labels_sample = df_train_sample['labels'].values

print(f"[2/3] Feature matrix shape: {X_sample.shape}")
print(f"      Data type: {X_sample.dtype}")
print(f"      Value range: [{X_sample.min():.4f}, {X_sample.max():.4f}]")

# Parse labels (convert space-separated indices to list)
print(f"[3/3] Label format validation:")
print(f"      Sample label (row 0): '{y_labels_sample[0]}'")

def parse_labels(label_str):
    if pd.isna(label_str) or label_str.strip() == '':
        return []
    return [int(idx.split(':')[0]) for idx in label_str.split()]

# Test label parsing
sample_labels_parsed = [parse_labels(lbl) for lbl in y_labels_sample[:5]]
print(f"      Parsed labels (first 5 rows):")
for i, lbl in enumerate(sample_labels_parsed):
    print(f"        Row {i}: {len(lbl)} labels - {lbl[:5]}{'...' if len(lbl) > 5 else ''}")

print("Sample data preprocessing test successful!")
print("Ready for Milestone 2: Binary Classifier Training")

Testing preprocessing on sample data
[1/3] Created sample with 100 rows
[2/3] Feature matrix shape: (100, 5000)
      Data type: float64
      Value range: [0.0000, 1.0000]
[3/3] Label format validation:
      Sample label (row 0): '832:1 1070:1 1337:1 1626:1 2971:1 3801:1'
      Parsed labels (first 5 rows):
        Row 0: 6 labels - [832, 1070, 1337, 1626, 2971]...
        Row 1: 4 labels - [855, 1848, 2701, 3083]
        Row 2: 6 labels - [207, 1109, 1419, 2079, 3102]...
        Row 3: 4 labels - [238, 637, 1358, 2742]
        Row 4: 6 labels - [710, 1623, 2325, 2437, 2564]...
Sample data preprocessing test successful!
Ready for Milestone 2: Binary Classifier Training


Save normalization parameters

In [11]:
# Save normalization parameters for future use
print("="*70)
print("Saving normalization parameters")
print("="*70)

# Save min/max parameters
import pickle

norm_params = {
    'feature_min': feature_min.to_dict(),
    'feature_max': feature_max.to_dict(),
    'feature_cols': feature_cols,
    'constant_features': constant_features[constant_features].index.tolist()
}

NORM_PARAMS_FILE = Path("normalization_params.pkl")
with open(NORM_PARAMS_FILE, 'wb') as f:
    pickle.dump(norm_params, f)

print(f"Saved normalization parameters to: {NORM_PARAMS_FILE}")
print(f"  - {len(feature_cols)} feature columns")
print(f"  - {num_constant} constant features")
print("These parameters will be used for:")
print("  Normalizing new data during inference")
print("  Ensuring consistency across distributed nodes")
print("  Reproducibility of preprocessing steps")

Saving normalization parameters
Saved normalization parameters to: normalization_params.pkl
  - 5000 feature columns
  - 0 constant features
These parameters will be used for:
  Normalizing new data during inference
  Ensuring consistency across distributed nodes
  Reproducibility of preprocessing steps


Categorical encoding: The original dataset was in .mat format where features were already stored as numeric matrices. Upon conversion to CSV, all 5000 feature columns retained their numeric form. No further categorical encoding was required.

Now implementing vertical data partioning/sharding

Since the dataset is "very wide" (5000 features), the bottleneck is feature dimensionality, not considering row count for now.
Each node receives all rows but a subset of feature columns, plus row_id and labels.
This allows each node to independently train binary classifiers on its feature slice.

In [12]:
# Cell 12: Partitioning setup and design
print("="*70)
print("Step 4: Vertical Data Partitioning (Column-Based Sharding)")
print("="*70)

import math
import os
import pickle
from pathlib import Path

# Load normalized training data
df_partitioned = pd.read_csv("eurlex_normalized.csv")

# Identify columns
all_cols = df_partitioned.columns.tolist()
feature_cols_part = [col for col in all_cols if col.startswith('f')]  # feature columns only
meta_cols = ['row_id', 'labels']  # always kept on every node

NUM_NODES = 4
total_features = len(feature_cols_part)
features_per_node = math.ceil(total_features / NUM_NODES)

print(f"[1/4] Partitioning strategy : Vertical column-based sharding")
print(f"[1/4] Total rows            : {len(df_partitioned):,}")
print(f"[1/4] Total feature columns : {total_features:,}")
print(f"[1/4] Number of nodes       : {NUM_NODES}")
print(f"[1/4] Features per node (≈) : {features_per_node:,}")
print(f"[1/4] Meta columns on every node: {meta_cols}")

Step 4: Vertical Data Partitioning (Column-Based Sharding)
[1/4] Partitioning strategy : Vertical column-based sharding
[1/4] Total rows            : 15,377
[1/4] Total feature columns : 5,000
[1/4] Number of nodes       : 4
[1/4] Features per node (≈) : 1,250
[1/4] Meta columns on every node: ['row_id', 'labels']


In [13]:
# Cell 13: Implement vertical sharding logic
print("[2/4] Implementing vertical sharding logic...")

PARTITION_DIR = Path("partitions")
PARTITION_DIR.mkdir(exist_ok=True)

def vertical_shard(df, feature_cols, num_nodes, meta_cols, output_dir):
    """
    Vertically partition dataframe into column-based shards.
    Each shard contains ALL rows, a subset of feature columns,
    plus meta_cols (row_id, labels) so each node can train independently.
    """
    partitions = []
    features_per_shard = math.ceil(len(feature_cols) / num_nodes)

    for node_id in range(num_nodes):
        start_col = node_id * features_per_shard
        end_col   = min(start_col + features_per_shard, len(feature_cols))

        shard_features = feature_cols[start_col:end_col]
        shard_cols     = meta_cols + shard_features       # row_id + labels + feature slice

        shard = df[shard_cols].copy()
        shard_path = output_dir / f"partition_{node_id}.csv"
        shard.to_csv(shard_path, index=False)

        partitions.append({
            'node_id'       : node_id,
            'start_col_idx' : start_col,
            'end_col_idx'   : end_col - 1,
            'num_features'  : len(shard_features),
            'num_rows'      : len(shard),
            'feature_range' : (shard_features[0], shard_features[-1]),
            'path'          : shard_path
        })

        print(f"[2/4] Node {node_id}: features {shard_features[0]}–{shard_features[-1]} "
              f"({len(shard_features):,} features, {len(shard):,} rows) → {shard_path}")

    return partitions

partitions = vertical_shard(df_partitioned, feature_cols_part, NUM_NODES, meta_cols, PARTITION_DIR)

[2/4] Implementing vertical sharding logic...
[2/4] Node 0: features f0–f1249 (1,250 features, 15,377 rows) → partitions\partition_0.csv
[2/4] Node 1: features f1250–f2499 (1,250 features, 15,377 rows) → partitions\partition_1.csv
[2/4] Node 2: features f2500–f3749 (1,250 features, 15,377 rows) → partitions\partition_2.csv
[2/4] Node 3: features f3750–f4999 (1,250 features, 15,377 rows) → partitions\partition_3.csv


In [14]:
# Cell 14: Verify workload balance across nodes
print("[3/4] Verifying workload balance across nodes...")

feat_counts = [p['num_features'] for p in partitions]
max_feat = max(feat_counts)
min_feat = min(feat_counts)
imbalance = ((max_feat - min_feat) / max_feat) * 100

print(f"\n[3/4] Feature distribution across nodes:")
for p in partitions:
    bar = "█" * (p['num_features'] // 50)
    print(f"       Node {p['node_id']}: {p['num_features']:,} features  "
          f"({p['feature_range'][0]} → {p['feature_range'][1]})  {bar}")

print(f"\n[3/4] Max features on any node : {max_feat:,}")
print(f"[3/4] Min features on any node : {min_feat:,}")
print(f"[3/4] Feature imbalance        : {imbalance:.2f}%")

if imbalance < 5:
    print("[3/4] ✓ Workload is well balanced across nodes")
else:
    print("[3/4] ⚠ Slight imbalance (last node gets fewer features if not divisible) — acceptable")

# Also confirm every node has the same number of rows
row_counts = [p['num_rows'] for p in partitions]
assert len(set(row_counts)) == 1, "Row count mismatch across partitions!"
print(f"[3/4] ✓ All nodes hold identical row count: {row_counts[0]:,}")

[3/4] Verifying workload balance across nodes...

[3/4] Feature distribution across nodes:
       Node 0: 1,250 features  (f0 → f1249)  █████████████████████████
       Node 1: 1,250 features  (f1250 → f2499)  █████████████████████████
       Node 2: 1,250 features  (f2500 → f3749)  █████████████████████████
       Node 3: 1,250 features  (f3750 → f4999)  █████████████████████████

[3/4] Max features on any node : 1,250
[3/4] Min features on any node : 1,250
[3/4] Feature imbalance        : 0.00%
[3/4] ✓ Workload is well balanced across nodes
[3/4] ✓ All nodes hold identical row count: 15,377


In [15]:
# Cell 15: Verify cross-node communication overhead
print("[4/4] Verifying cross-node communication overhead...")

# Load each shard and confirm:
# 1. No feature column appears in more than one partition
# 2. Every node has all rows (same row_ids)

all_feature_cols_seen = []
node_row_ids = []

for p in partitions:
    shard = pd.read_csv(p['path'])
    shard_feature_cols = [c for c in shard.columns if c.startswith('f')]
    all_feature_cols_seen.extend(shard_feature_cols)
    node_row_ids.append(set(shard['row_id'].tolist()))

# Check 1: no duplicate feature columns across nodes
total_features_seen = len(all_feature_cols_seen)
unique_features_seen = len(set(all_feature_cols_seen))

if total_features_seen == unique_features_seen:
    print("[4/4] ✓ No feature column appears in more than one partition")
    print("[4/4] ✓ Feature space is cleanly divided — zero redundancy")
else:
    print(f"[4/4] ⚠ Duplicate feature columns detected: "
          f"{total_features_seen - unique_features_seen} overlaps")

# Check 2: every node has exactly the same rows
reference_ids = node_row_ids[0]
all_match = all(ids == reference_ids for ids in node_row_ids)

if all_match:
    print("[4/4] ✓ All nodes hold exactly the same rows (required for vertical partitioning)")
else:
    print("[4/4] ⚠ Row ID mismatch across nodes — investigate!")

# Summary
print(f"\n[4/4] Partition summary:")
print(f"       Total unique features distributed : {unique_features_seen:,}")
print(f"       Rows on every node                : {len(reference_ids):,}")
print(f"       Cross-node communication overhead : NONE (no shared features, identical rows)")
print(f"       Partition directory               : {PARTITION_DIR}/")
print(f"       Files: {[f'partition_{i}.csv' for i in range(NUM_NODES)]}")

# Save partition metadata for Sameem's Dask framework
partition_meta = {
    'num_nodes'     : NUM_NODES,
    'meta_cols'     : meta_cols,
    'partitions'    : [
        {
            'node_id'      : p['node_id'],
            'path'         : str(p['path']),
            'num_features' : p['num_features'],
            'feature_range': p['feature_range']
        }
        for p in partitions
    ]
}
with open("partition_metadata.pkl", "wb") as f:
    pickle.dump(partition_meta, f)
print(f"\n[4/4] Partition metadata saved to partition_metadata.pkl")


[4/4] Verifying cross-node communication overhead...
[4/4] ✓ No feature column appears in more than one partition
[4/4] ✓ Feature space is cleanly divided — zero redundancy
[4/4] ✓ All nodes hold exactly the same rows (required for vertical partitioning)

[4/4] Partition summary:
       Total unique features distributed : 5,000
       Rows on every node                : 15,377
       Cross-node communication overhead : NONE (no shared features, identical rows)
       Partition directory               : partitions/
       Files: ['partition_0.csv', 'partition_1.csv', 'partition_2.csv', 'partition_3.csv']

[4/4] Partition metadata saved to partition_metadata.pkl


Now setting up the distributed environment for deploying the tasks to simulated worker nodes

In [ ]:
# Setting up LocalCluster
print("="*70)
print("Step 5: Distributed Framework Setup (Dask)")
print("="*70)

import pickle
import time
import pandas as pd
import numpy as np
from pathlib import Path

try:
    import dask
    import dask.dataframe as dd
    from dask.distributed import Client, LocalCluster, wait, as_completed
    print(f"[1/4] Dask version : {dask.__version__}")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "dask[distributed]", "-q"])
    import dask
    import dask.dataframe as dd
    from dask.distributed import Client, LocalCluster, wait, as_completed
    print(f"[1/4] Dask installed and imported. Version: {dask.__version__}")

# Load partition metadata produced by Fatima
with open("partition_metadata.pkl", "rb") as f:
    partition_meta = pickle.load(f)

NUM_NODES  = partition_meta["num_nodes"]       # 4
META_COLS  = partition_meta["meta_cols"]       # ["row_id", "labels"]
PARTITIONS = partition_meta["partitions"]      # list of dicts

print(f"[1/4] Loaded partition metadata")
print(f"      Nodes              : {NUM_NODES}")
print(f"      Meta columns       : {META_COLS}")
print(f"      Partition files    : {[p['path'] for p in PARTITIONS]}")

# Spin up a LocalCluster with one worker per partition (simulates 4 nodes)
print("[1/4] Starting Dask LocalCluster with 4 workers...")
cluster = LocalCluster(
    n_workers=NUM_NODES,
    threads_per_worker=1,   # 1 thread per worker = true parallelism
    memory_limit="2GB",
    silence_logs=True
)
client = Client(cluster)

print(f"[1/4] Cluster dashboard : {client.dashboard_link}")
print(f"[1/4] Workers registered : {len(client.scheduler_info()['workers'])}")
print("[1/4] Dask distributed framework ready.")

2026-04-01 22:50:45,769 - distributed.scheduler - INFO - State start


2026-04-01 22:50:45,821 - distributed.scheduler - INFO -   Scheduler at:     tcp://127.0.0.1:61597
2026-04-01 22:50:45,821 - distributed.scheduler - INFO -   dashboard at:  http://127.0.0.1:8787/status
2026-04-01 22:50:45,821 - distributed.scheduler - INFO - Registering Worker plugin shuffle


Step 5: Distributed Framework Setup (Dask)
[1/4] Dask version : 2026.3.0
[1/4] Loaded partition metadata
      Nodes              : 4
      Meta columns       : ['row_id', 'labels']
      Partition files    : ['partitions\\partition_0.csv', 'partitions\\partition_1.csv', 'partitions\\partition_2.csv', 'partitions\\partition_3.csv']
[1/4] Starting Dask LocalCluster with 4 workers...


2026-04-01 22:50:46,081 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:61600'
2026-04-01 22:50:46,088 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:61603'
2026-04-01 22:50:46,094 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:61601'
2026-04-01 22:50:46,103 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:61602'
2026-04-01 22:50:48,355 - distributed.scheduler - INFO - Register worker addr: tcp://127.0.0.1:61616 name: 2
2026-04-01 22:50:48,360 - distributed.scheduler - INFO - Starting worker compute stream, tcp://127.0.0.1:61616
2026-04-01 22:50:48,362 - distributed.core - INFO - Starting established connection to tcp://127.0.0.1:61618
2026-04-01 22:50:48,437 - distributed.scheduler - INFO - Register worker addr: tcp://127.0.0.1:61620 name: 0
2026-04-01 22:50:48,440 - distributed.scheduler - INFO - Starting worker compute stream, tcp://127.0.0.1:61620
2026-04-01 22:50:48,442 - distributed.core - IN

[1/4] Cluster dashboard : http://127.0.0.1:8787/status
[1/4] Workers registered : 4
[1/4] Dask distributed framework ready.


2026-04-01 22:51:12,042 - distributed.scheduler - INFO - Remove client Client-51119ef9-2df3-11f1-8e3c-cd56066bcd08
2026-04-01 22:51:12,044 - distributed.core - INFO - Received 'close-stream' from tcp://127.0.0.1:61628; closing.
2026-04-01 22:51:12,078 - distributed.scheduler - INFO - Remove client Client-51119ef9-2df3-11f1-8e3c-cd56066bcd08
2026-04-01 22:51:12,093 - distributed.scheduler - INFO - Close client connection: Client-51119ef9-2df3-11f1-8e3c-cd56066bcd08
2026-04-01 22:51:12,101 - distributed.scheduler - INFO - Retire worker addresses (stimulus_id='retire-workers-1775065872.1023088') (0, 1, 2, 3)
2026-04-01 22:51:12,145 - distributed.core - INFO - Received 'close-stream' from tcp://127.0.0.1:61624; closing.
2026-04-01 22:51:12,151 - distributed.core - INFO - Received 'close-stream' from tcp://127.0.0.1:61618; closing.
2026-04-01 22:51:12,153 - distributed.core - INFO - Received 'close-stream' from tcp://127.0.0.1:61627; closing.
2026-04-01 22:51:12,158 - distributed.scheduler 

In [21]:
#Defining worker-side preprocessing task and deploy to nodes
print("="*70)
print("[2/4] Deploying preprocessing tasks to worker nodes")
print("="*70)

# This function runs on the worker, not on the scheduler/client.
# Each worker receives its own partition path and processes it
# independently -- no data is shared between workers.

def worker_preprocess_task(partition_path: str, meta_cols: list) -> dict:
    """
    Worker-side preprocessing task.
    Loads a partition CSV, validates it, and returns a summary dict.
    This simulates what each distributed node would do in a real cluster.
    """
    import pandas as pd
    import numpy as np
    import os
    import time

    start = time.time()
    node_id = int(partition_path.split("partition_")[1].replace(".csv", ""))

    # 1. Load partition
    shard = pd.read_csv(partition_path)
    feature_cols = [c for c in shard.columns if c not in meta_cols]

    # 2. Validate: all features in [0, 1] after normalisation (Sohaib's work)
    feat_min = shard[feature_cols].min().min()
    feat_max = shard[feature_cols].max().max()
    in_range = (round(feat_min, 6) >= 0.0) and (round(feat_max, 6) <= 1.0)

    # 3. Validate: no missing values (Mustafa's guarantee)
    missing = int(shard[feature_cols].isnull().sum().sum())

    # 4. Compute sparsity
    zero_frac = float((shard[feature_cols] == 0).sum().sum()) / shard[feature_cols].size

    elapsed = time.time() - start
    return {
        "node_id"      : node_id,
        "pid"          : os.getpid(),
        "rows"         : len(shard),
        "features"     : len(feature_cols),
        "feat_min"     : round(feat_min, 6),
        "feat_max"     : round(feat_max, 6),
        "in_range"     : in_range,
        "missing"      : missing,
        "sparsity_pct" : round(zero_frac * 100, 2),
        "elapsed_sec"  : round(elapsed, 3),
    }

# Submit one task per partition to the cluster (non-blocking)
futures = []
for p in PARTITIONS:
    future = client.submit(
        worker_preprocess_task,
        p["path"],
        META_COLS,
        pure=False          # always re-run, do not cache
    )
    futures.append(future)
    print(f"  -> Dispatched task for Node {p['node_id']} ({p['path']})")

print(f"[2/4] {len(futures)} tasks dispatched to Dask workers.")
print("[2/4] Tasks are executing in parallel across workers...")

[2/4] Deploying preprocessing tasks to worker nodes
  -> Dispatched task for Node 0 (partitions\partition_0.csv)
  -> Dispatched task for Node 1 (partitions\partition_1.csv)
  -> Dispatched task for Node 2 (partitions\partition_2.csv)
  -> Dispatched task for Node 3 (partitions\partition_3.csv)
[2/4] 4 tasks dispatched to Dask workers.
[2/4] Tasks are executing in parallel across workers...


In [ ]:
#Monitoring task execution and collect results

print("="*70)
print("[3/4] Monitoring distributed task execution")
print("="*70)

results = []
dispatch_time = time.time()

# as_completed yields futures in the order they finish
print("Node | PID    | Rows   | Features | In-Range | Missing | Sparsity | Time(s)")
print("  " + "-"*75)

for future in as_completed(futures):
    r = future.result()
    results.append(r)
    status = "OK" if (r["in_range"] and r["missing"] == 0) else "WARN"
    print(f"  {r['node_id']}    | {r['pid']}  | "
          f"{r['rows']:,}  | {r['features']:,}     | "
          f"{str(r['in_range']):5}    | {r['missing']:,}      | "
          f"{r['sparsity_pct']}%  | {r['elapsed_sec']}s  [{status}]")

total_wall_time = round(time.time() - dispatch_time, 3)
results.sort(key=lambda x: x["node_id"])   # restore node order

print(f"[3/4] All {len(results)} worker tasks completed.")
print(f"[3/4] Total wall-clock time (parallel): {total_wall_time}s")

# Aggregate validation checks
all_in_range = all(r["in_range"] for r in results)
total_missing = sum(r["missing"] for r in results)
total_rows_processed = sum(r["rows"] for r in results)
total_features_processed = sum(r["features"] for r in results)

print(f"[3/4] Aggregate validation:")
print(f"      All partitions in [0,1] range : {all_in_range}")
print(f"      Total missing values           : {total_missing}")
print(f"      Total rows processed           : {total_rows_processed:,} "
      f"(= {len(results)} nodes × {results[0]['rows']:,} rows)")
print(f"      Total features processed       : {total_features_processed:,} "
      f"(= {len(results)} nodes × {results[0]['features']:,} features/node)")

[3/4] Monitoring distributed task execution
Node | PID    | Rows   | Features | In-Range | Missing | Sparsity | Time(s)
  ---------------------------------------------------------------------------
  3    | 6224  | 15,377  | 1,250     | True     | 0      | 95.23%  | 4.485s  [OK]
[3/4] All 4 worker tasks completed.
[3/4] Total wall-clock time (parallel): 0.057s
[3/4] Aggregate validation:
      All partitions in [0,1] range : True
      Total missing values           : 0
      Total rows processed           : 61,508 (= 4 nodes × 15,377 rows)
      Total features processed       : 5,000 (= 4 nodes × 1,250 features/node)


In [23]:
# Distributed system configuration document + shutdown

print("="*70)
print("[4/4] Distributed System Configuration Summary")
print("="*70)

import json

# Collect live scheduler info
scheduler_info = client.scheduler_info()
worker_info    = scheduler_info["workers"]

config_doc = {
    "framework"         : "Dask Distributed",
    "dask_version"      : dask.__version__,
    "cluster_type"      : "LocalCluster (simulated 4-node cluster)",
    "num_workers"       : len(worker_info),
    "threads_per_worker": 1,
    "memory_per_worker" : "2 GB",
    "partitioning"      : {
        "strategy"          : "Vertical (column-based) sharding",
        "num_partitions"    : NUM_NODES,
        "features_per_node" : results[0]["features"],
        "rows_per_node"     : results[0]["rows"],
        "cross_node_comm"   : "None (each node holds disjoint feature slice + all rows)"
    },
    "preprocessing_deployed": [
        {
            "node_id"      : r["node_id"],
            "pid"          : r["pid"],
            "status"       : "OK" if (r["in_range"] and r["missing"] == 0) else "WARN",
            "elapsed_sec"  : r["elapsed_sec"],
            "sparsity_pct" : r["sparsity_pct"]
        }
        for r in results
    ],
    "total_wall_time_sec" : total_wall_time,
    "validation" : {
        "all_in_range"  : all_in_range,
        "total_missing" : total_missing
    }
}

# Print formatted configuration
print(json.dumps(config_doc, indent=4))

# Save to disk for reproducibility
CONFIG_FILE = Path("distributed_config.json")
with open(CONFIG_FILE, "w") as f:
    json.dump(config_doc, f, indent=4)
print(f"[4/4] Configuration saved to: {CONFIG_FILE}")

# Graceful shutdown of the cluster
client.close()
cluster.close()
print("[4/4] Dask cluster shut down cleanly.")
print("" + "="*70)
print("Milestone 1 COMPLETE")
print("  Mustafa  -> Data ingestion & validation     [DONE]")
print("  Sohaib   -> Normalisation & encoding        [DONE]")
print("  Fatima   -> Vertical partitioning           [DONE]")
print("  Sameem   -> Dask framework & deployment     [DONE]")
print("="*70)
print("Output artefacts:")
print("  eurlex_normalized.csv          (Sohaib)")
print("  eurlex_test_normalized.csv     (Sohaib)")
print("  normalization_params.pkl       (Sohaib)")
print("  partitions/partition_0..3.csv  (Fatima)")
print("  partition_metadata.pkl         (Fatima)")
print("  distributed_config.json        (Sameem)")
print("Ready for Milestone 2: Parallel Binary Classifier Training.")

2026-04-01 22:51:12,101 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:61600'. Reason: nanny-close
2026-04-01 22:51:12,109 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-01 22:51:12,113 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:61601'. Reason: nanny-close
2026-04-01 22:51:12,119 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-01 22:51:12,121 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:61602'. Reason: nanny-close
2026-04-01 22:51:12,129 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-01 22:51:12,131 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:61603'. Reason: nanny-close
2026-04-01 22:51:12,135 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close


[4/4] Distributed System Configuration Summary
{
    "framework": "Dask Distributed",
    "dask_version": "2026.3.0",
    "cluster_type": "LocalCluster (simulated 4-node cluster)",
    "num_workers": 4,
    "threads_per_worker": 1,
    "memory_per_worker": "2 GB",
    "partitioning": {
        "strategy": "Vertical (column-based) sharding",
        "num_partitions": 4,
        "features_per_node": 1250,
        "rows_per_node": 15377,
        "cross_node_comm": "None (each node holds disjoint feature slice + all rows)"
    },
    "preprocessing_deployed": [
        {
            "node_id": 0,
            "pid": 36380,
            "status": "OK",
            "elapsed_sec": 4.499,
            "sparsity_pct": 94.97
        },
        {
            "node_id": 1,
            "pid": 28884,
            "status": "OK",
            "elapsed_sec": 4.52,
            "sparsity_pct": 95.48
        },
        {
            "node_id": 2,
            "pid": 15720,
            "status": "OK",
         

2026-04-01 22:51:12,580 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:61602' closed.
2026-04-01 22:51:12,588 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:61600' closed.
2026-04-01 22:51:12,597 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:61601' closed.
2026-04-01 22:51:12,614 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:61603' closed.


[4/4] Dask cluster shut down cleanly.
Milestone 1 COMPLETE
  Mustafa  -> Data ingestion & validation     [DONE]
  Sohaib   -> Normalisation & encoding        [DONE]
  Fatima   -> Vertical partitioning           [DONE]
  Sameem   -> Dask framework & deployment     [DONE]
Output artefacts:
  eurlex_normalized.csv          (Sohaib)
  eurlex_test_normalized.csv     (Sohaib)
  normalization_params.pkl       (Sohaib)
  partitions/partition_0..3.csv  (Fatima)
  partition_metadata.pkl         (Fatima)
  distributed_config.json        (Sameem)
Ready for Milestone 2: Parallel Binary Classifier Training.
